# CivicSite Permit Copilot - demo notebook
Elastic (hybrid search, filters, aggregations) + Mistral (embeddings in Elastic, function calling, structured output) over NYC DOB records.
Run `civicsite fetch/build/setup/index` first.

In [ ]:
from civicsite.config import get_settings
from civicsite.store import get_store
import json
s = get_settings(); store = get_store(s, 'elastic')
print(store.backend, store.stats())

## 1. Elastic: mapping (semantic_text backed by a Mistral inference endpoint)

In [ ]:
m = store.es.indices.get_mapping(index=s.idx_records)
props = m[s.idx_records]['mappings']['properties']
print(json.dumps({k: props[k] for k in ['description','description_semantic','bin','event_date']}, indent=1))

## 2. Entity resolution: address -> BIN/BBL

In [ ]:
cands = store.resolve('439 East 77 Street'); [(c['bin'], c['bbl'], c['address'], c['risk_score']) for c in cands]

## 3. Unified timeline + deterministic, cited flags

In [ ]:
a = store.analysis(cands[0]['bin'])
for f in a['flags'][:5]:
    print(f['severity'].upper(), f['reason_code'], '|', f['title'])
    print('   next:', f['recommended_action'])
    for e in f['evidence']: print('   cite:', e['record_id'], e['source_url'])

## 4. Hybrid search: BM25 + Mistral semantic via RRF (meaning, not keywords)

In [ ]:
r = store.search('landlord converted apartments illegally without approval', record_type='violation', size=5)
print(r['mode'])
for h in r['hits']: print(h['record_id'], '|', h['address'], '|', (h.get('description') or '')[:110])

## 5. Aggregations: where is applicant-owned delay concentrated?

In [ ]:
p = store.portfolio(limit=5)
print(p['by_reason']); print(p['by_severity'])
[(b['address'], b['risk_score'], b['top_flag_title']) for b in p['top']]

## 6. Mistral agent: function calling over Elastic + rule checks

In [ ]:
from civicsite.agent import ask
r = ask(store, s, 'What is blocking 439 East 77 Street and who has to act?')
for t in r['tool_trace']: print('->', t['tool'], t['args'], t['summary'])
print(r['answer']); print(r['citations'])

## 7. Mistral structured output with verbatim-quote verification

In [ ]:
from civicsite.extract import extract_record
rec = store.get_record(a['flags'][0]['evidence'][0]['record_id'])
d = extract_record(store, s, rec)
print(json.dumps(d.get('extracted'), indent=1)); print('quotes verified:', d.get('quotes_verified'))